In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_NSIT_Dwarka_Delhi_CPCB_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,358.0,178.0,220.0,170.0,227.0,310.0,147.0,50.0,115.0,156.0,336.0,290.0
1,2,355.0,194.0,112.0,233.0,291.0,234.0,157.0,65.0,72.0,164.0,302.0,238.0
2,3,352.0,165.0,145.0,169.0,353.0,181.0,125.0,68.0,68.0,NaN,413.0,273.0
3,4,366.0,286.0,186.0,244.0,378.0,256.0,61.0,68.0,68.0,170.0,414.0,212.0
4,5,321.0,113.0,202.0,222.0,328.0,270.0,106.0,61.0,53.0,133.0,409.0,183.0
5,6,277.0,130.0,198.0,258.0,307.0,204.0,63.0,55.0,61.0,129.0,449.0,179.0
6,7,322.0,236.0,270.0,277.0,329.0,267.0,52.0,56.0,53.0,128.0,348.0,176.0
7,8,345.0,152.0,259.0,277.0,206.0,270.0,53.0,42.0,56.0,145.0,321.0,218.0
8,9,347.0,140.0,261.0,259.0,162.0,172.0,91.0,51.0,81.0,187.0,362.0,145.0
9,10,285.0,338.0,270.0,269.0,147.0,168.0,169.0,58.0,99.0,142.0,347.0,147.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,358.0,178.000000,220.000000,170.000000,227.000000,310.000000,147.0,50.000000,115.000000,156.000000,336.000000,290.000000
1,2,355.0,194.000000,112.000000,233.000000,291.000000,234.000000,157.0,65.000000,72.000000,164.000000,302.000000,238.000000
2,3,352.0,165.000000,145.000000,169.000000,353.000000,181.000000,125.0,68.000000,68.000000,199.971429,413.000000,273.000000
3,4,366.0,286.000000,186.000000,244.000000,378.000000,256.000000,61.0,68.000000,68.000000,170.000000,414.000000,212.000000
4,5,321.0,113.000000,202.000000,222.000000,328.000000,270.000000,106.0,61.000000,53.000000,133.000000,409.000000,183.000000
5,6,277.0,130.000000,198.000000,258.000000,307.000000,204.000000,63.0,55.000000,61.000000,129.000000,449.000000,179.000000
6,7,322.0,236.000000,270.000000,277.000000,329.000000,267.000000,52.0,56.000000,53.000000,128.000000,348.000000,176.000000
7,8,345.0,152.000000,259.000000,277.000000,206.000000,270.000000,53.0,42.000000,56.000000,145.000000,321.000000,218.000000
8,9,347.0,140.000000,261.000000,259.000000,162.000000,172.000000,91.0,51.000000,81.000000,187.000000,362.000000,145.000000
9,10,285.0,338.000000,270.000000,269.000000,147.000000,168.000000,169.0,58.000000,99.000000,142.000000,347.000000,147.000000
